# Data Cleaning

This notebook cleans and standardizes the educational access and outcome
datasets collected from BPS.

The main objectives are:
- standardize column names and formats,
- flatten multi-level headers,
- retain province-level observations,
- standardize province names,
- filter relevant years,
- convert variables into appropriate data types,
- identify and handle missing or invalid values,
- validate the cleaned datasets before integration.

In [74]:
import pandas as pd
import os

PROCESSED_PATH = "../datasets/processed/"

schools = pd.read_csv(
    os.path.join(PROCESSED_PATH, "schools_2025_processed.csv")
)

libraries = pd.read_csv(
    os.path.join(PROCESSED_PATH, "libraries_2025_processed.csv")
)

reading_fondness = pd.read_csv(
    os.path.join(PROCESSED_PATH, "reading_fondness_2025_processed.csv")
)

eys = pd.read_csv(
    os.path.join(PROCESSED_PATH, "eys_2025_processed.csv")
)

completion = pd.read_csv(
    os.path.join(PROCESSED_PATH, "completion_2023_processed.csv")
)

facilities = pd.read_csv(
    os.path.join(PROCESSED_PATH, "facilities_2025_processed.csv")
)

education_25plus = pd.read_csv(
    os.path.join(PROCESSED_PATH, "education_25plus_2025_processed.csv")
)

literacy = pd.read_csv(
    os.path.join(PROCESSED_PATH, "literacy_2025_processed.csv")
)

In [76]:
datasets = {
    "schools": schools,
    "libraries": libraries,
    "reading_fondness": reading_fondness,
    "completion": completion,
    "facilities": facilities,
    "eys": eys,
    "education_25plus": education_25plus,
    "literacy": literacy
}

for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

schools
Shape: (42, 10)
Columns: ['province', 'schools_primary_public', 'schools_primary_private', 'schools_primary_total', 'teachers_primary_public', 'teachers_primary_private', 'teachers_primary_total', 'pupils_primary_public', 'pupils_primary_private', 'pupils_primary_total']
libraries
Shape: (39, 18)
Columns: ['province', 'special_library_a', 'special_library_b', 'special_library_c', 'special_library_total', 'school_library_a', 'school_library_b', 'school_library_c', 'school_library_total', 'academic_library_a', 'academic_library_b', 'academic_library_c', 'academic_library_total', 'public_library_a', 'public_library_b', 'public_library_c', 'public_library_total', 'public_library_all']
reading_fondness
Shape: (39, 5)
Columns: ['province', 'reading_fondness_level', 'tgm_pre_reading', 'tgm_reading', 'tgm_post_reading']
completion
Shape: (39, 4)
Columns: ['province', 'completion_elementary', 'completion_junior_high', 'completion_senior_high']
facilities
Shape: (39, 6)
Columns: ['provin

In [77]:
def normalize_province(df, column="province"):
    df = df.copy()

    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    return df

In [78]:
schools = normalize_province(schools)
libraries = normalize_province(libraries)
reading_fondness = normalize_province(reading_fondness)
completion = normalize_province(completion)
facilities = normalize_province(facilities)

In [79]:
PROVINCES = (
    facilities.loc[
        facilities["province"] != "INDONESIA",
        "province"
    ]
    .dropna()
    .unique()
)

print("Number of provinces:", len(PROVINCES))

Number of provinces: 38


In [80]:
province_mapping = {
    "KEPULAUAN BANGKA BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN RIAU": "KEP. RIAU"
}

In [81]:
province_datasets = {
    "schools": schools,
    "libraries": libraries,
    "reading_fondness": reading_fondness,
    "completion": completion,
    "facilities": facilities
}

for df in province_datasets.values():
    df["province"] = df["province"].replace(province_mapping)

In [82]:
for name, df in province_datasets.items():

    matched = df["province"].isin(PROVINCES).sum()

    print(
        f"{name:20} "
        f"rows={len(df):3} | "
        f"matched={matched:3}"
    )

schools              rows= 42 | matched= 38
libraries            rows= 39 | matched= 38
reading_fondness     rows= 39 | matched= 38
completion           rows= 39 | matched= 38
facilities           rows= 39 | matched= 38


In [83]:
for name in province_datasets:

    province_datasets[name] = province_datasets[name][
        province_datasets[name]["province"].isin(PROVINCES)
    ].copy()

In [84]:
schools = province_datasets["schools"]
libraries = province_datasets["libraries"]
reading_fondness = province_datasets["reading_fondness"]
completion = province_datasets["completion"]
facilities = province_datasets["facilities"]

In [85]:
for name, df in province_datasets.items():

    print(
        f"{name:20} "
        f"rows={len(df):3} | "
        f"unique provinces={df['province'].nunique():3}"
    )

schools              rows= 38 | unique provinces= 38
libraries            rows= 38 | unique provinces= 38
reading_fondness     rows= 38 | unique provinces= 38
completion           rows= 38 | unique provinces= 38
facilities           rows= 38 | unique provinces= 38


In [97]:
eys["region_clean"] = (
    eys["region"]
    .astype("string")
    .str.strip()
)

In [98]:
eys["province_key"] = (
    eys["region_clean"]
    .str.upper()
)

In [99]:
eys_province_mapping = {
    "KEPULAUAN RIAU": "KEP. RIAU",
    "D I YOGYAKARTA": "DI YOGYAKARTA"
}

In [100]:
eys["province_key"] = eys["province_key"].replace(
    eys_province_mapping
)

In [101]:
eys["is_province"] = (
    eys["region_clean"] == eys["region_clean"].str.upper()
)

In [103]:
eys_province = eys[
    eys["is_province"]
    & eys["province_key"].isin(PROVINCES)
].copy()

In [104]:
print("EYS province rows:", len(eys_province))
print("Unique provinces:", eys_province["province_key"].nunique())

EYS province rows: 38
Unique provinces: 38


In [105]:
eys_province = eys_province.rename(
    columns={
        "province_key": "province"
    }
)

In [106]:
eys_province = eys_province.drop(
    columns=[
        "region",
        "region_clean",
        "is_province"
    ]
)

In [107]:
print(eys_province.shape)
print(eys_province.columns.tolist())
print(eys_province.head())

(38, 3)
['eys_male', 'eys_female', 'province']
   eys_male eys_female        province
0     14.31      14.63            ACEH
24    13.35      13.75  SUMATERA UTARA
58    13.89      14.87  SUMATERA BARAT
78    13.34      13.71            RIAU
91    13.19      13.67           JAMBI


In [110]:
education_25plus["region_clean"] = (
    education_25plus["region"]
    .astype("string")
    .str.strip()
)

In [111]:
education_25plus["province_key"] = (
    education_25plus["region_clean"]
    .str.upper()
)

In [112]:
education_25plus_mapping = {
    "KEPULAUAN RIAU": "KEP. RIAU",
    "D I YOGYAKARTA": "DI YOGYAKARTA"
}

In [113]:
education_25plus["province_key"] = (
    education_25plus["province_key"]
    .replace(education_25plus_mapping)
)

In [120]:
education_25plus["is_province"] = (
    education_25plus["region_clean"]
    == education_25plus["region_clean"].str.upper()
)

In [121]:
education_25plus_province = education_25plus[
    education_25plus["is_province"]
    & education_25plus["province_key"].isin(PROVINCES)
].copy()

In [122]:
education_25plus_province = education_25plus_province.rename(
    columns={
        "province_key": "province"
    }
)

education_25plus_province = education_25plus_province.drop(
    columns=[
        "region",
        "region_clean",
        "is_province"
    ]
)

In [127]:
literacy["province"] = (
    literacy["Province"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [129]:
literacy_province_mapping = {
    "KEPULAUAN BANGKA BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN RIAU": "KEP. RIAU"
}

In [130]:
literacy["province"] = literacy["province"].replace(
    literacy_province_mapping
)

In [132]:
literacy_2025 = literacy[
    (literacy["province"].isin(PROVINCES))
    & (literacy["Urban_Rural"] == "Urban+Rural")
].copy()

In [134]:
literacy_2025 = literacy_2025[
    [
        "province",
        "Male_2025",
        "Female_2025",
        "Male+Female_2025"
    ]
].copy()

In [136]:
literacy_2025 = literacy_2025.rename(
    columns={
        "Male_2025": "literacy_male",
        "Female_2025": "literacy_female",
        "Male+Female_2025": "literacy_total"
    }
)

In [140]:
CLEANED_PATH = "../datasets/cleaned/"
os.makedirs(CLEANED_PATH, exist_ok=True)

In [141]:
schools.to_csv(
    os.path.join(CLEANED_PATH, "schools_2025_clean.csv"),
    index=False
)

libraries.to_csv(
    os.path.join(CLEANED_PATH, "libraries_2025_clean.csv"),
    index=False
)

reading_fondness.to_csv(
    os.path.join(CLEANED_PATH, "reading_fondness_2025_clean.csv"),
    index=False
)

completion.to_csv(
    os.path.join(CLEANED_PATH, "completion_rate_2023_clean.csv"),
    index=False
)

facilities.to_csv(
    os.path.join(CLEANED_PATH, "facilities_2025_clean.csv"),
    index=False
)

eys_province.to_csv(
    os.path.join(CLEANED_PATH, "eys_2025_clean.csv"),
    index=False
)

education_25plus_province.to_csv(
    os.path.join(CLEANED_PATH, "education_25plus_2025_clean.csv"),
    index=False
)

literacy_2025.to_csv(
    os.path.join(CLEANED_PATH, "literacy_2025_clean.csv"),
    index=False
)